# Psycholinguistic Marker Extraction with RoBERTa-LoRA
## Token Classification for Conspiracy Marker Detection

This notebook trains a token classification model to extract psycholinguistic markers:
- **Marker Types**: Action, Actor, Effect, Evidence, Victim
- **Model**: RoBERTa-base with LoRA fine-tuning
- **Approach**: BIO tagging scheme for multi-label NER

In [ ]:
# Environment setup
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
import numpy as np

print('='*60)
print('Environment')
print('='*60)
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('='*60)

In [ ]:
# Imports
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from transformers import (
    RobertaTokenizerFast, RobertaForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print('✓ Imports complete')

In [ ]:
# Load data
BASE = Path('../..')
TRAIN_FILE = BASE / 'train_rehydrated.jsonl'

def load_data(file_path):
    data = []
    with open(file_path) as f:
        for line in f:
            try:
                data.append(json.loads(line))
            except:
                pass
    return data

train_data = load_data(TRAIN_FILE)
print(f'✓ Loaded {len(train_data)} examples')
print(f'Sample: {len(train_data[0].get("markers", []))} markers')

In [ ]:
# Label mapping - BIO tagging
MARKER_TYPES = ['Action', 'Actor', 'Effect', 'Evidence', 'Victim']

label_list = ['O']
for mt in MARKER_TYPES:
    label_list.extend([f'B-{mt}', f'I-{mt}'])

label_to_id = {l: i for i, l in enumerate(label_list)}
id_to_label = {i: l for l, i in label_to_id.items()}
num_labels = len(label_list)

print(f'Labels: {num_labels} ({len(MARKER_TYPES)} marker types with BIO tagging)')

In [ ]:
# Tokenizer and alignment
MODEL_NAME = 'roberta-base'
MAX_LENGTH = 256

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME, add_prefix_space=True)

def tokenize_and_align_labels(examples):
    tok = tokenizer(
        examples['text'], truncation=True, max_length=MAX_LENGTH,
        return_offsets_mapping=True, is_split_into_words=False
    )
    labels, all_markers = [], examples.get('markers', [])
    for i, offsets in enumerate(tok['offset_mapping']):
        ex_labels = [label_to_id['O']] * len(offsets)
        markers = sorted(all_markers[i] if i < len(all_markers) else [], key=lambda x: x['startIndex'])
        for m in markers:
            b_lbl, i_lbl = label_to_id.get(f"B-{m['type']}"), label_to_id.get(f"I-{m['type']}")
            if b_lbl is None: continue
            first = True
            for ti, (s, e) in enumerate(offsets):
                if s is None: continue
                if s < m['endIndex'] and e > m['startIndex']:
                    ex_labels[ti] = b_lbl if first else (i_lbl if ex_labels[ti] == label_to_id['O'] else ex_labels[ti])
                    first = False
        labels.append(ex_labels)
    tok['labels'] = labels
    return tok

print(f'✓ Tokenizer ready')

In [ ]:
# Train/val split
VAL_SPLIT = 0.1
train_idx, val_idx = train_test_split(range(len(train_data)), test_size=VAL_SPLIT, random_state=42)
train_ds = Dataset.from_list([train_data[i] for i in train_idx])
val_ds = Dataset.from_list([train_data[i] for i in val_idx])

print(f'Tokenizing...')
train_ds = train_ds.map(tokenize_and_align_labels, batched=True, remove_columns=['_id', 'text', 'markers', 'subreddit', 'conspiracy', 'annotator'])
val_ds = val_ds.map(tokenize_and_align_labels, batched=True, remove_columns=['_id', 'text', 'markers', 'subreddit', 'conspiracy', 'annotator'])

print(f'✓ Train: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
# Model configuration
from transformers import RobertaConfig

config = RobertaConfig.from_pretrained(MODEL_NAME)
config.num_labels = num_labels
config.id2label = id_to_label
config.label2id = label_to_id

model = RobertaForTokenClassification.from_pretrained(MODEL_NAME, config=config)

if torch.cuda.is_available():
    model = model.to('cuda')
    print('✓ Model on GPU')

if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()

# LoRA
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.1
TARGET_MODULES = ['query', 'value', 'key', 'dense']

lora_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS, r=LORA_R, lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT, target_modules=TARGET_MODULES, bias='none'
)
model = get_peft_model(model, lora_config)

total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✓ LoRA applied: {train_p:,}/{total_p:,} params ({100*train_p/total_p:.2f}%)')

In [ ]:
# Metrics
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=2)
    true_l, pred_l = [], []
    for p, l in zip(preds, labels):
        for pi, li in zip(p, l):
            if li != -100:
                true_l.append(id_to_label[li])
                pred_l.append(id_to_label[pi])
    f1_micro = f1_score(true_l, pred_l, average='micro')
    ent_l = [l for l in true_l if l != 'O']
    ent_p = [p for p, l in zip(pred_l, true_l) if l != 'O']
    f1_ent = f1_score(ent_l, ent_p, average='micro') if ent_l else 0.0
    return {'f1_overall': f1_micro, 'f1_entity': f1_ent}

print('✓ Metrics defined')

In [ ]:
# Training
BATCH_SIZE, GRAD_ACC, LR, EPOCHS = 8, 2, 3e-4, 10
WEIGHT_DECAY, WARMUP = 0.01, 0.1

OUTPUT_DIR = BASE / 'Markers-Extraction' / 'models' / 'roberta-base-markers-lora'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

use_bf16 = False
if torch.cuda.is_available():
    try:
        _ = torch.tensor([1.0], dtype=torch.bfloat16)
        use_bf16 = True
    except:
        pass

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR), learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='f1_entity', greater_is_better=True,
    logging_steps=50, report_to='none', seed=42,
    gradient_accumulation_steps=GRAD_ACC, bf16=use_bf16,
    gradient_checkpointing=True, save_total_limit=2
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True)
early_stop = EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)

trainer = Trainer(
    model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
    tokenizer=tokenizer, data_collator=data_collator,
    compute_metrics=compute_metrics, callbacks=[early_stop]
)

print('='*60)
print('Starting Training')
print('='*60)
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_result = trainer.train()
eval_result = trainer.evaluate()

print('\n' + '='*60)
print('Training Complete!')
print('='*60)
print(f'Loss: {train_result.training_loss:.4f}')
for k, v in eval_result.items():
    if k.startswith('eval_'):
        print(f'{k}: {v:.4f}')
print('='*60)

In [ ]:
# Evaluation
print('Generating predictions...')
predictions = trainer.predict(val_ds)
pred_labels = np.argmax(predictions.predictions, axis=2)
true_labels = predictions.label_ids

true_flat, pred_flat = [], []
for p, l in zip(pred_labels, true_labels):
    for pi, li in zip(p, l):
        if li != -100:
            true_flat.append(id_to_label[li])
            pred_flat.append(id_to_label[pi])

print('\n' + '='*60)
print('Classification Report (Validation)')
print('='*60)
print(classification_report(true_flat, pred_flat))
print('='*60)

In [ ]:
# Save model
final_dir = OUTPUT_DIR / 'final_model'
final_dir.mkdir(exist_ok=True)

trainer.model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

label_mapping = {'label_to_id': label_to_id, 'id_to_label': id_to_label, 'marker_types': MARKER_TYPES}
with open(final_dir / 'label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)

print(f'✓ Model saved to: {final_dir}')
print('✓ Training complete!')